# Label database exports from Google Drive

This notebook prepares database exports for Rayyan, suggests research labels, and saves results to a private Google Drive folder. It uses code pinned to commit `12112a782d0a4d049d6991177bb0f2cade529874`. A standard CPU runtime is sufficient; no model API or database API credentials are needed.

## Researcher workflow at a glance

1. Set up a dedicated Google account and a private project folder in Drive.
2. Run the documented database searches through your institutional access, recording queries, filters, dates, and hit counts.
3. Download full citation records with abstracts and all available fields in the formats below.
4. Put the downloads in `QMatrix/exports/<database>/`, keeping queries and batches separate.
5. Run this notebook in order; optionally edit the custom-labeling cell before suggesting labels.
6. Check export counts, metadata coverage, and label suggestions; keep the audit and original files.
7. Follow the instructions at the end of the notebook to upload your chosen files to Rayyan and verify the import.

## Before you begin: keep the project separate

Ideally, create a separate, dedicated Google account (a “dummy” project account) for running this Colab notebook, and keep only the project notebook, inputs, and outputs in that account's Drive. This keeps the workflow separate from personal files and limits the files exposed to the scripts. Sign in to Colab with that account, save a copy of this notebook in its Drive, and authorize that account when connecting Drive.

This is account-level separation: the standard Colab Drive mount allows notebook code to access the files in the mounted account's Drive. Choosing one project folder does not limit the mount's permissions. [Google's Drive-access explanation](https://research.google.com/colaboratory/faq.html#drive-mount).

## Create the necessary database files

Use the [database-specific searches, queries, filters, and download instructions](https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/12112a782d0a4d049d6991177bb0f2cade529874/docs_agent/manual_search_links.md). Sign in through your library or institution where required. The queries use the documented **2021–2027** retrieval filter; apply the review's first-online-date cutoff and eligibility criteria during screening.

For each query, record the exact query, selected database/collection, filters, search date, and displayed result count **before downloading**. Export every result, using successive batches if the interface limits each download. Include abstracts, keywords, identifiers, and all available bibliographic details. Inspect a small export first to confirm abstracts are present.

| Database | Search and download preparation | File for this notebook |
| --- | --- | --- |
| Scopus | Advanced document search; use the documented query. Select results → Export → RIS; include bibliographic fields, abstracts, and keywords. | `.ris` |
| Web of Science | Core Collection → Advanced Search/Query Builder. Export results/ranges as RIS with Full Record and Cited References where available, including abstracts. | `.ris` |
| APA PsycInfo | Select APA PsycInfo in your institution's EBSCO interface; use Advanced Search and apply the documented date filter. Export all batches as RIS with abstracts and all available fields. | `.ris` |
| ERIC | Select ERIC in EBSCO; use Advanced Search and apply the documented date filter. Export all batches as RIS with abstracts and all available fields. | `.ris` |
| PubMed | Run the documented query, then Save → All results → PubMed format → Create file. Split into nonoverlapping date ranges if the export limit is reached. | `.nbib` or tagged PubMed `.txt` |
| IEEE Xplore | Run Search queries 1–3 separately in All Metadata with the documented filters. Export each query's results/batches as RIS; confirm abstracts are included. | `.ris` |
| ACM DL | Run Search queries 1–3 separately with the documented collection/metadata settings and filters. Export selected results/batches using EndNote; inspect the file for abstracts and other fields. | `.enw` |

Keep the original downloads unchanged. This notebook reads RIS, NBIB, EndNote, and recognized tagged `.txt` files; it does not parse provider-specific CSV, BibTeX, XML, or API JSON. Keep supplementary downloads outside `exports/`.

## Arrange the files in Drive

Create a folder such as `QMatrix` in the dedicated account's My Drive:

```text
QMatrix/
  exports/
    Scopus/query1_batch1.ris
    Web of Science/query1_batch1.ris
    APA PsycInfo/query1_batch1.ris
    ERIC/query1_batch1.ris
    PubMed/query1_batch1.nbib
    IEEE Xplore/query1_batch1.ris
    IEEE Xplore/query2_batch1.ris
    IEEE Xplore/query3_batch1.ris
    ACM DL/query1_batch1.enw
    ACM DL/query2_batch1.enw
    ACM DL/query3_batch1.enw
  export_counts.csv   # Optional expected counts for each exported batch.
  input.csv           # Optional previous candidate CSV for comparison.
```

Add further batches as needed. Keep a separate folder for each database; those folder names become the records' database provenance. Multiple queries can retrieve the same article; the notebook retains those records for later duplicate resolution.

Only `exports/` and its database folders are required. For count checking, create `export_counts.csv` with actual counts from your search log, one row per exported file:

```csv
file,expected_count
Scopus/query1_batch1.ris,500
PubMed/query1_batch1.nbib,120
```

These counts are examples; replace them with your own. Filenames are relative to `exports/`. A batch's expected count is the number in that batch, not the total search count. Record zero-result searches in your search log; no export file is needed for them.

In “Choose the private data folder,” set `DATA_ROOT` to your Drive project folder. Set `COUNTS_FILE = "export_counts.csv"` and/or `BASELINE_FILE = "input.csv"` if using those optional checks; otherwise leave them blank. Then run the remaining cells in order. Count mismatches stop labeling and save the audit for investigation.

More details: [Colab workflow](https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/default/docs_agent/colab.md) and [data-sharing guidance](https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/12112a782d0a4d049d6991177bb0f2cade529874/docs_agent/data_sharing.md). Keep database exports and derived records private unless the applicable licenses permit sharing.


## Load the code


In [ ]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import sys
import tempfile

CODE_URL = "https://github.com/gettodaze/OMSCS_ITR_QMatrix.git"
CODE_REF = "12112a782d0a4d049d6991177bb0f2cade529874"  # Exact tested code commit; do not replace with a branch name.
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REF):
    raise ValueError("CODE_REF must be a full 40-character Git commit SHA.")
CODE_ROOT = Path(tempfile.mkdtemp(prefix="qmatrix-code-")) / "repo"
subprocess.run(["git", "init", str(CODE_ROOT)], check=True, capture_output=True, text=True)
subprocess.run(["git", "remote", "add", "origin", CODE_URL], cwd=CODE_ROOT,
               check=True, capture_output=True, text=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", CODE_REF], cwd=CODE_ROOT,
               check=True, capture_output=True, text=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=CODE_ROOT,
               check=True, capture_output=True, text=True)
CODE_REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CODE_ROOT, text=True).strip()
if CODE_REVISION != CODE_REF:
    raise RuntimeError("Checked-out code does not match the pinned revision.")
sys.path.insert(0, str(CODE_ROOT))
for name in ("parse_records", "prepare_rayyan", "label_exports"):
    sys.modules.pop(name, None)
print("Code revision:", CODE_REVISION)


## Connect Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## Choose the private data folder

Create `QMatrix/exports/<database>/` in My Drive and put the native downloads in the database folders. Supported files: `.ris`, `.nbib`, `.enw`, and tagged PubMed/RIS/EndNote `.txt`. The old candidate CSV is optional, used only for comparison. Raw provider CSV/BibTeX files are not parsed by this workflow.


In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/QMatrix")  # Edit this if your folder is elsewhere.
EXPORT_ROOT = DATA_ROOT / "exports"
DRIVE_OUTPUT_ROOT = DATA_ROOT / "output"
BASELINE_FILE = ""  # Optional, e.g. "input.csv", relative to DATA_ROOT.
COUNTS_FILE = ""    # Optional, e.g. "export_counts.csv", relative to DATA_ROOT.

if not EXPORT_ROOT.is_dir():
    raise FileNotFoundError("Create the exports/<database>/ folders in your chosen Drive data folder first.")
for name in (BASELINE_FILE, COUNTS_FILE):
    if name and not (DATA_ROOT / name).is_file():
        raise FileNotFoundError(DATA_ROOT / name)
print("Data folder selected. Original downloads will remain unchanged.")


## Prepare native files and check counts


In [ ]:
from datetime import datetime, timezone
from prepare_rayyan import prepare

RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S_%f_UTC")
WORK_ROOT = Path(tempfile.mkdtemp(prefix="qmatrix-data-"))
SNAPSHOT = WORK_ROOT / "exports"
shutil.copytree(EXPORT_ROOT, SNAPSHOT)

def local_optional(name):
    if not name:
        return None
    destination = Path(tempfile.mkdtemp(prefix="optional-", dir=WORK_ROOT)) / Path(name).name
    shutil.copy2(DATA_ROOT / name, destination)
    return destination

PREPARED_ROOT = WORK_ROOT / "prepared"
audit = prepare(SNAPSHOT, PREPARED_ROOT,
                baseline=local_optional(BASELINE_FILE), counts=local_optional(COUNTS_FILE))
print("Exported records:", sum(item["records"] for item in audit["files"]))
if any(item["count_matches"] is False for item in audit["files"]):
    failed_output = DRIVE_OUTPUT_ROOT / (RUN_ID + "_count_mismatch")
    shutil.copytree(PREPARED_ROOT, failed_output)
    raise ValueError(f"Export counts do not match. Audit and native files saved to {failed_output}; check before labeling/importing.")


## Optional: customize labels

Uncomment the function and assignment below to change labeling in this notebook. It receives a typed `ExportRecord` with all common citation fields, provenance, and every original tag/value in `source_fields`. The record and its source fields are read-only. Return a list of existing `Label` members, custom strings, or both.

Under the hood: [record type, default adapter and output writer](https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/12112a782d0a4d049d6991177bb0f2cade529874/label_exports.py) · [default label categories and matching rules](https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/12112a782d0a4d049d6991177bb0f2cade529874/parse_records.py).

To rerun after editing, rerun “Prepare native files and check counts” first, then this cell and the remaining cells, to create fresh output directories.


In [ ]:
from label_exports import ExportRecord, default_labels
from parse_records import Label

# Complete record, default adapter, and output writing:
# https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/12112a782d0a4d049d6991177bb0f2cade529874/label_exports.py
# Default label categories, regex rules, and text-matching behavior:
# https://github.com/gettodaze/OMSCS_ITR_QMatrix/blob/12112a782d0a4d049d6991177bb0f2cade529874/parse_records.py

custom_labeler = None  # Keep the default behavior unless you uncomment below.

# def custom_labels(record: ExportRecord) -> list[Label | str]:
#     labels: list[Label | str] = list(default_labels(record))
#
#     # Example using the normalized document type.
#     if record.document_type in ("CONF", "CPAPER", "Conference Paper"):
#         labels.append("conference paper")
#
#     # Example using PubMed's original MeSH fields.
#     if any("Mental Disorders" in term for term in record.source_fields.get("MH", ())):
#         labels.append(Label.PSYCHOLOGICAL_CLINICAL)
#
#     # Other fields include record.notes, journal, language, authors,
#     # keywords, year, doi, database, and all original source_fields.
#     # Replace this function's body to replace the default rules completely.
#     return labels
#
# custom_labeler = custom_labels


## Suggest labels


In [ ]:
from label_exports import label_exports

LABELS_ROOT = WORK_ROOT / "labels"
summary = label_exports(PREPARED_ROOT / "originals", LABELS_ROOT, labeler=custom_labeler)
assert summary["records"] == sum(item["records"] for item in audit["files"])
print("Labeled records:", summary["records"])
print("Records missing abstracts:", summary["without_abstract"])
print("Records without a suggested label:", summary["without_suggested_labels"])


## Save results to Drive


In [ ]:
RUN_OUTPUT = DRIVE_OUTPUT_ROOT / RUN_ID
RUN_OUTPUT.mkdir(parents=True, exist_ok=False)
shutil.copytree(PREPARED_ROOT, RUN_OUTPUT / "prepared")
shutil.copytree(LABELS_ROOT, RUN_OUTPUT / "labels")
(RUN_OUTPUT / "run_metadata.json").write_text(json.dumps({
    "code_url": CODE_URL, "code_ref": CODE_REF, "code_revision": CODE_REVISION,
    "run_id": RUN_ID, "baseline_used": bool(BASELINE_FILE), "counts_used": bool(COUNTS_FILE),
    "labeling_method": summary["labeling_method"], "custom_labeling": custom_labeler is not None,
    "records": summary["records"]
}, indent=2) + "\n", encoding="utf-8")
print("Saved results to:", RUN_OUTPUT)


## Review aggregate results


In [ ]:
print("Records by database:")
for name, count in summary["by_database"].items():
    print(f"  {name}: {count}")
print("Records by suggested label:")
for name, count in summary["by_label"].items():
    print(f"  {name}: {count}")


## After processing: review and upload to Rayyan

The “Save results to Drive” cell prints the location of your new run folder under `QMatrix/output/`. Open that folder in Drive, keep the originals and audit, and download the files you choose to import to your computer.

### 1. Review the run

Check `prepared/audit.json` against your search log: file and database counts, document types, years, and missing abstracts. Inspect `labels/label_summary.json` and `labels/labelled_records.csv` to review the suggested labels. Labels are preliminary suggestions; confirm them during research screening. The notebook keeps query/database overlaps and does not deduplicate or decide eligibility.

Keep `run_metadata.json` and your edited notebook together so collaborators can identify the code revision and reproduce any custom labeling logic.

### 2. Choose one set of files to import

| Files | What they provide |
| --- | --- |
| `prepared/by_database/<database>.ris`, `.nbib`, or `.enw` | Native citation metadata, concatenated by database and format, suitable for named Rayyan imports. Generated label suggestions are in the separate labeling outputs. |
| `labels/labelled_records.csv` | Common citation fields, suggested labels in their own column and notes, provenance, and original tags in extra columns. Test the mapping before using it for the whole review. |

For the simplest metadata import, use `prepared/by_database/` and retain the labeled CSV as your review companion. If you choose the labeled CSV, verify in a small test that the title, abstract, DOI, authors, notes, and any desired labels arrive correctly. Rayyan may ignore extra columns, and automatic creation of Rayyan label objects from the suggested-label column or notes has not been verified.

Use one import set for the review. Uploading both the native files and the labeled CSV adds the same citations twice. The optional `prepared/combined.*` files also contain the same records as `by_database/`; choose either the combined files or the database-specific files.

### 3. Upload and verify

1. Open or create the appropriate review in [Rayyan](https://www.rayyan.ai/).
2. Before the full import, try 10–20 records in a separate test review and inspect their fields.
3. In the target review, select **Add References** from the Overview or Review data tab.
4. Choose upload from your device, select your chosen files, and give each import a clear source/date name, such as “Scopus — 2026-10-09.” Use your actual search date.
5. Click **Continue** and allow processing to finish.
6. In **Review data → Imported References**, compare the imported count with the exported count in your audit. For native files, use the database/format or constituent file counts; for the combined CSV, use the labeling summary's total. Investigate differences before screening.
7. Spot-check abstracts, titles, authors, DOI, and notes. If using the labeled CSV, confirm how suggested labels are represented; apply or adjust Rayyan labels as needed.
8. Resolve duplicates within Rayyan using the review team's agreed process, then begin screening. Keep original search counts and duplicate counts for the review's reporting.

The notebook prepares files; you perform this upload in Rayyan. See [Rayyan's upload instructions](https://help.rayyan.ai/hc/en-us/articles/17216953358353-How-to-Import-References-into-Rayyan) and [supported formats](https://help.rayyan.ai/hc/en-us/articles/4406426903825-Supported-File-Formats-for-Importing-into-Rayyan).

### Retain the research artifacts

Keep `prepared/originals/`, `prepared/audit.json`, both labeling outputs, `label_summary.json`, `run_metadata.json`, your search log, and the edited notebook in the private project folder. Retain these even if Rayyan normalizes or omits some provider-specific fields. Share executed notebooks containing record previews only with the authorized research team; omit code-cell outputs when distributing the notebook more widely.
